# Aula 4 — População por estado e por município (Censo 2022 x 2010)

Passo a passo: leitura do arquivo, agregação por estado, agregação por município e exportação em CSV.

## 1-3. Bibliotecas e caminho do arquivo
(pasta do projeto já criada e planilha já copiada para dentro dela)

In [ ]:
import pandas as pd
import openpyxl

caminho = "CD2022_Populacao_2010_Compatibilizada_20231222.xlsx"

## 4. Leitura da tabela

O arquivo do IBGE tem 2 linhas de título antes do cabeçalho de verdade, então usamos `skiprows=2`. Também existe uma coluna vazia no início e, no rodapé, duas linhas de nota/fonte que precisam ser descartadas.

In [ ]:
# leitura "crua", igual ao que já tínhamos: mostra o título e o cabeçalho bagunçados
arquivo_bruto = pd.read_excel(caminho, sheet_name="Municípios")
arquivo_bruto.head()

In [ ]:
# leitura limpa: pula as 2 linhas de título e usa a 3ª linha como cabeçalho
df = pd.read_excel(caminho, sheet_name="Municípios", skiprows=2)

# remove a coluna vazia à esquerda
df = df.drop(columns=["Unnamed: 0"])

# remove as linhas de nota/fonte do rodapé (não têm código de UF)
df = df.dropna(subset=["COD. UF"]).reset_index(drop=True)

# renomeia colunas para nomes mais simples
df = df.rename(columns={
    "COD. UF": "cod_uf",
    "COD. MUNIC": "cod_munic",
    "NOME DO MUNICÍPIO": "municipio",
    "População Município 2010\n(Sinopse)": "pop_2010_sinopse",
    "População 2010 (Alterações de Limites até 2022)1": "pop_2010",
    "População Censo 2022": "pop_2022",
})

df.head()

**Observação:** usamos a coluna `pop_2010` ("População 2010 - Alterações de Limites até 2022") em vez de `pop_2010_sinopse`. É a própria nota do IBGE no rodapé da planilha que recomenda isso: essa versão do dado de 2010 já foi recalculada dentro dos limites municipais de 2022, para que a comparação com 2022 não seja distorcida por mudanças de fronteira entre municípios.

## 5. Tabela de população agregada por estado

In [ ]:
pop_por_estado = (
    df.groupby("UF")[["pop_2010", "pop_2022"]]
    .sum()
    .reset_index()
)

pop_por_estado.head()

## 6. Diferença de população e ordenação por crescimento (2010 → 2022)

In [ ]:
pop_por_estado["crescimento_2010_2022"] = pop_por_estado["pop_2022"] - pop_por_estado["pop_2010"]
pop_por_estado["crescimento_pct"] = (
    pop_por_estado["crescimento_2010_2022"] / pop_por_estado["pop_2010"] * 100
).round(2)

pop_por_estado = pop_por_estado.sort_values("crescimento_2010_2022", ascending=False).reset_index(drop=True)
pop_por_estado

## 7. Salvar a tabela por estado em CSV

In [ ]:
pop_por_estado.to_csv("populacao_por_estado_2010_2022.csv", index=False)

## 8. Agora por município

In [ ]:
pop_por_municipio = (
    df.groupby(["UF", "municipio"])[["pop_2010", "pop_2022"]]
    .sum()
    .reset_index()
)

pop_por_municipio.head()

## 9. Diferença de população e ordenação por crescimento (2010 → 2022)

In [ ]:
pop_por_municipio["crescimento_2010_2022"] = pop_por_municipio["pop_2022"] - pop_por_municipio["pop_2010"]
pop_por_municipio["crescimento_pct"] = (
    pop_por_municipio["crescimento_2010_2022"] / pop_por_municipio["pop_2010"] * 100
).round(2)

pop_por_municipio = pop_por_municipio.sort_values("crescimento_2010_2022", ascending=False).reset_index(drop=True)
pop_por_municipio.head(15)

## 10. Salvar a tabela por município em CSV

In [ ]:
pop_por_municipio.to_csv("populacao_por_municipio_2010_2022.csv", index=False)